In [0]:
%pip install openpyxl

In [0]:
import json, re
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CATALOG= "hawkerreach"
VOL= f"/Volumes/{CATALOG}/bronze/raw_files"

# table: (file in volume, source url, date downloaded)
SOURCES= {
    "raw_hawker_centres":  ("hawker_centres.geojson", "https://data.gov.sg/datasets/d_4a086da0a5553be1d89383cd90d07ecd/view", "2026-10-17"),
    "raw_closures":        ("closures.csv",           "https://data.gov.sg/datasets/d_bda4baa634dd1cc7a6c7cad5f19e2d68/view", "2026-10-17"),
    "raw_hdb_buildings":   ("hdb_buildings.geojson",  "https://data.gov.sg/datasets/d_16b157c52ed637edd6ba1232e026258d/view", "2026-10-17"),
    "raw_hdb_property":    ("hdb_property.csv",       "https://data.gov.sg/datasets/d_17f5382f26140b1fdae0ba2ef6239d2f/view", "2026-10-17"),
    "raw_subzones":        ("subzones.geojson",       "https://data.gov.sg/datasets/d_8594ae9ff96d0c708bc2af633048edfb/view", "2026-10-17"),
    "raw_planning_areas":  ("planning_areas.geojson", "https://data.gov.sg/datasets/d_4765db0e87b9c86336792efe8a1f7a66/view", "2026-10-17"),
    #"raw_population_2026": ("population_2026.xlsx",   "https://www.singstat.gov.sg/find-data/search-by-theme/population/geographic-distribution", "2026-10-17"),
}

def clean_cols(pdf):
    pdf.columns= [re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_").lower() for c in pdf.columns]
    return pdf

def read_geojson(path):
    with open(path) as f:
        gj= json.load(f)
    rows= []
    for feat in gj["features"]:
        row= {k: (None if v is None else str(v)) for k, v in feat["properties"].items()}
        row["geometry"]= json.dumps(feat["geometry"])  # raw geometry kept as GeoJSON text
        rows.append(row)
    return pd.DataFrame(rows)

def read_file(path):
    if path.endswith(".geojson"):
        return read_geojson(path)
    if path.endswith(".csv"):
        return pd.read_csv(path, dtype=str, keep_default_na=False)  # keeps "nil"/"NA" as text
    if path.endswith(".xlsx"):
        sheets= pd.read_excel(path, sheet_name=None, header=2, dtype=str, keep_default_na=False)
        return pd.concat([df.assign(sheet=name) for name, df in sheets.items()], ignore_index=True)
    raise ValueError(f"unknown file type: {path}")

for table, (fname, url, dl_date) in SOURCES.items():
    pdf= clean_cols(read_file(f"{VOL}/{fname}"))
    pdf= pdf.astype(object).where(pdf.notna(), None)
    schema= StructType([StructField(c, StringType(), True) for c in pdf.columns])
    sdf= (spark.createDataFrame(pdf, schema=schema)
          .withColumn("source_url", F.lit(url))
          .withColumn("downloaded_at", F.lit(dl_date).cast("date"))
          .withColumn("ingested_at", F.current_timestamp()))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.bronze.{table}")
    print(f"{table}: {sdf.count()} rows")